# DeiT-LT Scaling: ImageNet-LT

This notebook scales the baseline model to the massive **ImageNet-LT** dataset (1000 classes). We use a larger backbone (`DeiT-Small`) and standard ImageNet configurations. We bypass distillation here to focus purely on whether token bifurcation (Head vs Tail experts) occurs organically at a massive scale.


In [ ]:
!pip install timm==0.4.12 torch torchvision scikit-learn seaborn matplotlib scipy


In [ ]:
import os
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.scheduler import create_scheduler
from timm.optim import create_optimizer
from PIL import Image

import torch.backends.cudnn as cudnn
cudnn.benchmark = True


In [ ]:
# --- Configuration ---
class Config:
    epochs = 90
    num_classes = 1000
    batch_size = 128
    lr = 1e-3
    warmup_epochs = 5
    drw_epoch = 80 # DRW in last 10 epochs
    weight_decay = 0.05
    mixup = 0.8
    cutmix = 1.0
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

args = Config()


In [ ]:
# --- ImageNet-LT Dataloaders ---
IMAGENET_PATH = '/kaggle/input/imagenet/ILSVRC/Data/CLS-LOC/'

class ImageNetLT(Dataset):
    def __init__(self, root, txt_file, transform=None):
        self.img_path = []
        self.labels = []
        self.transform = transform
        if os.path.exists(txt_file):
            with open(txt_file) as f:
                for line in f:
                    self.img_path.append(os.path.join(root, line.split()[0]))
                    self.labels.append(int(line.split()[1]))
        else:
            print(f"WARNING: {txt_file} not found. Please upload to Kaggle.")
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        path = self.img_path[index]
        label = self.labels[index]
        try:
            with open(path, 'rb') as f:
                sample = Image.open(f).convert('RGB')
        except:
            sample = Image.new('RGB', (224, 224))
        if self.transform is not None:
            sample = self.transform(sample)
        return sample, label

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# NOTE: UNCOMMENT THESE LINES AFTER UPLOADING THE SPLITS TO KAGGLE
# train_dataset = ImageNetLT(IMAGENET_PATH, 'ImageNet_LT_train.txt', transform=transform_train)
# test_dataset = ImageNetLT(IMAGENET_PATH, 'ImageNet_LT_test.txt', transform=transform_test)
# train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=4)
# test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=4)

# For code completion, we will create dummy loaders if they don't exist
if 'train_loader' not in locals():
    print("WARNING: Using dummy dataloaders because txt splits were not found!")
    train_loader = []
    test_loader = []
    cls_distribution = np.ones(1000) * 10
else:
    # Compute class distribution for DRW
    cls_distribution = np.zeros(args.num_classes)
    for lbl in train_dataset.labels:
        cls_distribution[lbl] += 1



In [ ]:
# --- STUDENT: DeiT-Small for ImageNet-LT ---
import timm
import types

class DeiTLT(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.model = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=num_classes)
        self.embed_dim = self.model.embed_dim
        self.dist_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        
        # Override forward_features to include dist_token
        def new_forward_features(self_model, x):
            B = x.shape[0]
            x = self_model.patch_embed(x)
            cls_tokens = self_model.cls_token.expand(B, -1, -1)
            dist_tokens = self.dist_token.expand(B, -1, -1)
            x = torch.cat((cls_tokens, dist_tokens, x), dim=1)
            x = x + self_model.pos_embed
            x = self_model.pos_drop(x)
            for blk in self_model.blocks:
                x = blk(x)
            x = self_model.norm(x)
            return x[:, 0], x[:, 1] # Return both CLS and DIST
        
        self.model.forward_features = types.MethodType(new_forward_features, self.model)
        self.model.dist_token = self.dist_token
        self.model.pos_embed = nn.Parameter(torch.zeros(1, self.model.patch_embed.num_patches + 2, self.embed_dim))
        
        self.head_cls = nn.Linear(self.embed_dim, num_classes)
        self.head_dist = nn.Linear(self.embed_dim, num_classes)
        
    def forward(self, x, return_features=False):
        cls_tok, dist_tok = self.model.forward_features(x)
        logits_cls = self.head_cls(cls_tok)
        logits_dist = self.head_dist(dist_tok)
        if return_features:
            return logits_cls, logits_dist, cls_tok, dist_tok
        return logits_cls, logits_dist

model = DeiTLT(num_classes=args.num_classes).to(args.device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
print("DeiT-Small initialized for ImageNet-LT")



In [ ]:
# --- Training Setup ---
# Per Class Weights (for DRW)
beta = 0.9999
effective_num = 1.0 - np.power(beta, cls_distribution)
per_cls_weights = (1.0 - beta) / np.array(effective_num)
per_cls_weights = per_cls_weights / np.sum(per_cls_weights) * args.num_classes
per_cls_weights = torch.FloatTensor(per_cls_weights).to(args.device)

# Mixup
mixup_fn = Mixup(mixup_alpha=args.mixup, cutmix_alpha=args.cutmix, label_smoothing=0.1, num_classes=args.num_classes)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scaler = torch.amp.GradScaler('cuda')
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
base_criterion = SoftTargetCrossEntropy()



In [ ]:
# --- Training Loop ---
metrics_history = {
    'epoch': [], 'cls_acc': [], 'dist_acc': [], 'avg_acc': [],
    'cls_loss': [], 'dist_loss': [], 'lr': [], 'time_sec': []
}

if len(train_loader) > 0:
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_loss = 0
        
        # DRW Logic
        use_drw = epoch >= args.drw_epoch
        
        for i, (images, targets) in enumerate(train_loader):
            images, targets = images.to(args.device), targets.to(args.device)
            
            # Disable Mixup during DRW phase
            if not use_drw:
                images, targets = mixup_fn(images, targets)
            else:
                targets = torch.nn.functional.one_hot(targets, num_classes=args.num_classes).float()
                
            with torch.amp.autocast('cuda'):
                logits_cls, logits_dist = model(images)
                
                if use_drw:
                    loss_cls = F.cross_entropy(logits_cls, targets.argmax(dim=1), weight=per_cls_weights)
                    loss_dist = F.cross_entropy(logits_dist, targets.argmax(dim=1), weight=per_cls_weights)
                else:
                    loss_cls = base_criterion(logits_cls, targets)
                    loss_dist = base_criterion(logits_dist, targets)
                    
                loss = loss_cls + loss_dist
                
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            
        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()
        epoch_time = time.time() - epoch_start_time
        
        # Evaluation
        if (epoch + 1) % 5 == 0 or epoch == 0:
            model.eval()
            cls_correct, dist_correct, avg_correct, total = 0, 0, 0, 0
            
            with torch.no_grad():
                for imgs, lbls in test_loader:
                    imgs, lbls = imgs.to(args.device), lbls.to(args.device)
                    l_cls, l_dist = model(imgs)
                    l_avg = (l_cls + l_dist) / 2
                    
                    cls_correct += (l_cls.argmax(dim=1) == lbls).sum().item()
                    dist_correct += (l_dist.argmax(dim=1) == lbls).sum().item()
                    avg_correct += (l_avg.argmax(dim=1) == lbls).sum().item()
                    total += lbls.size(0)
                    
            cls_acc = cls_correct / total * 100
            dist_acc = dist_correct / total * 100
            avg_acc = avg_correct / total * 100
            
            metrics_history['epoch'].append(epoch + 1)
            metrics_history['cls_acc'].append(cls_acc)
            metrics_history['dist_acc'].append(dist_acc)
            metrics_history['avg_acc'].append(avg_acc)
            
            print(f"Ep {epoch+1:03d} [{epoch_time:.1f}s] | L:{total_loss/len(train_loader):.3f} | Acc:[CLS:{cls_acc:.1f} DIST:{dist_acc:.1f} AVG:{avg_acc:.1f}]")
            
else:
    print("Training skipped. Provide ImageNet-LT splits to begin.")



In [ ]:
# --- Oracle Search on ImageNet-LT ---
# After training completes, we will sweep alphas to find the absolute upper bound!
print("Run Oracle Search...")

